In [1]:
# !wget https://gist.githubusercontent.com/huseinzol05/98974ae8c6c7a65d4bc0af9f5003786a/raw/2e06e71ef7349a57bc58cc9913ae6bae1f9f8447/mp.py

In [2]:
from tqdm import tqdm
from huggingface_hub import HfFileSystem
from glob import glob
from multiprocess import Pool
import soundfile as sf
import pandas as pd
import polars as pl
import os
import re
import string
import mp
import json
import itertools


def chunks(l, n, folder):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n, folder)


def multiprocessing(strings, folder, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores, folder)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

In [3]:
folders = [
    'ASR-PART3-Train',
    'SQA-PART3-Train',
    'SDS-PART3-Train',
    'PQA-AR-Dialogue-Train',
    'PQA-GR-Dialogue-Train',
]

In [4]:
fs = HfFileSystem()

folder = folders[0]
files = fs.ls(f"datasets/MERaLiON/Multitask-National-Speech-Corpus-v1/{folder}", detail=False)
files = sorted(files)
filtered_files = [f for f in files if '.parquet' in f]

In [5]:
!rm -rf {folder} {folder}-audio

In [6]:
def loop(files):
    files, _, folder = files
    folder_audio = f'{folder}-audio'
    os.makedirs(folder, exist_ok = True)
    os.makedirs(folder_audio, exist_ok = True)
    fs = HfFileSystem()
    for f in tqdm(files):
        results = []
        f_only = os.path.split(f)[1]
        f_only_done = os.path.join(folder, f_only)
        f_ = f_only.replace('.parquet', '')
        
        if os.path.exists(f_only_done):
            try:
                with open(f_only_done) as fopen:
                    json.load(fopen)
                continue
            except:
                pass
        
        try:
            if not os.path.exists(f_only):
                fs.get(f, f_only)
            df = pl.read_parquet(f_only)
        except:
            fs.get(f, f_only)
            df = pl.read_parquet(f_only)
            
        for i in range(len(df)):
            row = df[i].to_dict(as_series = False)
            
            q = row['instruction'][0]
            a = row['answer'][0]
                
            audio_filename = os.path.join(folder_audio, f'{f_}-{i}.mp3')
            if not os.path.exists(audio_filename):
                with open(audio_filename, 'wb') as fopen:
                    fopen.write(row['context'][0]['bytes'])
            results.append({
                'instruction': q,
                'answer': a,
                'audio_filename': audio_filename
            })
        
        with open(f_only_done, 'w') as fopen:
            json.dump(results, fopen)
        
        os.remove(f_only)
        
        del df

In [10]:
multiprocessing(filtered_files, folder, loop, cores = 10, returned = False)

In [9]:
for folder in folders[1:]:
    files = fs.ls(f"datasets/MERaLiON/Multitask-National-Speech-Corpus-v1/{folder}", detail=False)
    files = sorted(files)
    filtered_files = [f for f in files if '.parquet' in f]
    multiprocessing(filtered_files, folder, loop, cores = 10, returned = False)